# OCRFix-RU — Case Study 2.3

**Hybrid Word+Character N-gram for Error Correction**

This notebook reproduces the repository experiments: synthetic OCR noise, comparison of the hybrid corrector against a word-only n-gram baseline, Accuracy / WER / CER metrics, and ablations over $\alpha$ and noise level.

## How to run

- Install the package from the repo root: `python -m pip install -e .` (recommended), **or** rely on the next cell — it prepends `src/` to `sys.path`.
- Working directory: project root **or** `notebooks/` (the setup cell finds the root via `pyproject.toml`).
- To regenerate `artifacts/report.json` and the poster TeX: from the root run `python scripts/build_case_study.py` (the Gutenberg corpus is downloaded if missing).

In [ ]:
"""Use local ocrfix_ru without requiring pip install -e (optional)."""
from __future__ import annotations

import json
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = _here if (_here / "pyproject.toml").exists() else _here.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ocrfix_ru.dataset import load_corpus_sentences
from ocrfix_ru.evaluate import (
    evaluate_models_on_synthetic_ocr,
    run_ablation_study,
)
from ocrfix_ru.evaluate import WordOnlyCorrector
from ocrfix_ru.corrector import HybridCorrector
from ocrfix_ru.noise import add_ocr_noise

ARTIFACTS = ROOT / "artifacts" / "report.json"
CORPUS = ROOT / "data" / "raw" / "russian_corpus.txt"

print("ROOT =", ROOT)
print("Corpus present:", CORPUS.exists(), "|", CORPUS)
print("Saved report present:", ARTIFACTS.exists(), "|", ARTIFACTS)

## 1. Motivation and setup

After OCR, characters are often confused with visually similar glyphs (Cyrillic ↔ Latin/digits). Tokens fall out of vocabulary coverage, and a **word-only** n-gram model is weaker where plausible **letter** restoration matters.

**Baseline:** word n-gram corrector with limited editing (`WordOnlyCorrector` — no OCR confusion substitutions in candidate generation).

**Method:** `HybridCorrector` — candidates driven by typical OCR replacements plus joint reranking with word LM and char LM (scalar $\alpha$ weights lexical context vs character model).

## 2. Data and synthetic noise

Noise is deterministic for a fixed `seed`: each character may be dropped, replaced from an OCR confusion table, or kept. This yields a reproducible benchmark without labeled scans.

Below: the same clean string with two different seeds.

In [ ]:
clean = "мама мыла раму а кот спит на окне"
rate = 0.25
for s in (1, 2):
    noisy = add_ocr_noise(clean, error_rate=rate, seed=s)
    print(f"seed={s}: {noisy!r}")

## 3. Main experiment (headline numbers)

Hyperparameters match `scripts/build_case_study.py`: fixed `seed`, number of noisy trials, noise intensity, and hybrid $\alpha$. With `base_sentences=None`, training texts are the small built-in phrase list inside `evaluate_models_on_synthetic_ocr`, replicated so count-based n-gram LMs have enough mass.

**Metrics:**
- **Accuracy** — fraction of reference token positions where the corrected token matches (index-aligned).
- **WER** — word-level edit distance between reference and hypothesis, normalized by reference word count.
- **CER** — same at character level.

In [ ]:
HEADLINE = dict(seed=42, sample_size=80, error_rate=0.30, alpha=0.90)

main_report = evaluate_models_on_synthetic_ocr(**HEADLINE)

acc_w = main_report["word_only_accuracy"]
acc_h = main_report["hybrid_accuracy"]
delta_acc = main_report["delta"]

print("Settings:", HEADLINE)
print()
print(f"{'Metric':<22} {'Word-only':>12} {'Hybrid':>12} {'Delta (hybrid - word)':>22}")
print("-" * 72)
print(
    f"{'Accuracy':<22} {acc_w:12.4f} {acc_h:12.4f} {delta_acc:+22.4f}"
)
print(
    f"{'WER':<22} {main_report['word_only_wer']:12.4f} {main_report['hybrid_wer']:12.4f} "
    f"{main_report['hybrid_wer'] - main_report['word_only_wer']:+22.4f}"
)
print(
    f"{'CER':<22} {main_report['word_only_cer']:12.4f} {main_report['hybrid_cer']:12.4f} "
    f"{main_report['hybrid_cer'] - main_report['word_only_cer']:+22.4f}"
)
print()
print(
    "Relative Accuracy gain:",
    f"{(delta_acc / acc_w * 100):+.2f}%" if acc_w else "n/a",
)

In [ ]:
# Cross-check against artifacts/report.json when the build script has already run
if ARTIFACTS.exists():
    saved = json.loads(ARTIFACTS.read_text(encoding="utf-8"))["main_report"]
    keys = sorted(main_report.keys())
    ok = all(abs(main_report[k] - saved[k]) < 1e-9 for k in keys)
    print("Matches artifacts/report.json (main_report):", "yes" if ok else "no")
    if not ok:
        for k in keys:
            a, b = main_report[k], saved[k]
            if abs(a - b) >= 1e-9:
                print(f"  {k}: notebook={a} json={b}")
else:
    print("No artifacts/report.json — skipping cross-check.")

## 4. Qualitative example (single sentence)

Train both models on a short phrase list, inject noise into one reference sentence, and compare corrections.

In [ ]:
demo_sentences = [
    "мама мыла раму",
    "кот спит на окне",
    "текст содержит шум после оцифровки",
    "простая модель исправляет ошибки",
]
train = demo_sentences * 20

hybrid = HybridCorrector(word_n=2, char_n=4, alpha=0.9)
word_only = WordOnlyCorrector(word_n=2, char_n=4)
hybrid.fit(train)
word_only.fit(train)

ref = "мама мыла раму и кот спит"
noisy = add_ocr_noise(ref, error_rate=0.35, seed=123)
print("Reference:", ref)
print("Noisy:    ", noisy)
print("Word-only:", word_only.correct_text(noisy))
print("Hybrid:   ", hybrid.correct_text(noisy))

## 5. Ablation study

Grid matches `build_case_study.py`: multiple `seed` values, `error_rate` levels, and $\alpha$. Each configuration reports mean accuracy and error metrics.

- By default we **load rows from `artifacts/report.json`** so graders need not wait for a full recompute.
- Set `RUN_FULL_ABLATION = True` to **recompute from scratch** (requires `data/raw/russian_corpus.txt`; obtain via `python scripts/build_case_study.py` or manual download).

In [ ]:
RUN_FULL_ABLATION = False  # True = full grid (slow); requires CORPUS

ABLATION_PARAMS = dict(
    seeds=[7, 11, 19],
    error_rates=[0.1, 0.2, 0.3],
    alphas=[0.0, 0.3, 0.7, 1.0],
    sample_size=24,
)

if RUN_FULL_ABLATION:
    if not CORPUS.exists():
        raise FileNotFoundError(
            f"Missing corpus at {CORPUS}. Run scripts/build_case_study.py or add the file manually."
        )
    base = load_corpus_sentences(CORPUS, min_words=3, max_sentences=1800)
    ablation_rows = run_ablation_study(**ABLATION_PARAMS, base_sentences=base)
    print(f"Recomputed ablation rows: {len(ablation_rows)}")
else:
    if not ARTIFACTS.exists():
        raise FileNotFoundError(
            "Missing artifacts/report.json. Build artifacts first or set RUN_FULL_ABLATION with a corpus."
        )
    ablation_rows = json.loads(ARTIFACTS.read_text(encoding="utf-8"))["ablation"]
    print(f"Loaded ablation rows from JSON: {len(ablation_rows)}")

In [ ]:
# Top configurations by Accuracy delta (hybrid - word_only)
sorted_rows = sorted(
    ablation_rows,
    key=lambda r: r["delta_accuracy"],
    reverse=True,
)[:8]

hdr = f"{'seed':>5} {'eps':>6} {'alpha':>6} {'dAcc':>10} {'hybrid Acc':>12} {'word Acc':>10}"
print("Top 8 by Accuracy delta")
print(hdr)
print("-" * len(hdr))
for r in sorted_rows:
    print(
        f"{int(r['seed']):5d} {r['error_rate']:6.2f} {r['alpha']:6.2f} "
        f"{r['delta_accuracy']:+10.4f} {r['hybrid_accuracy']:12.4f} {r['word_only_accuracy']:10.4f}"
    )

In [ ]:
# Mean Accuracy delta by noise level (average over seeds and alphas in the current table)
from collections import defaultdict

by_eps = defaultdict(list)
for r in ablation_rows:
    by_eps[r["error_rate"]].append(r["delta_accuracy"])

print("Mean Accuracy delta by error_rate:")
for eps in sorted(by_eps):
    xs = by_eps[eps]
    print(f"  eps={eps:.1f}: mean={sum(xs)/len(xs):+.4f} (n={len(xs)})")

## 6. Conclusions

- Under the **controlled** synthetic OCR protocol, the **word+char hybrid** improves Accuracy and lowers **WER/CER** versus the word-only baseline.
- At **low** noise and when lexical scores dominate ($\alpha \to 1$), hybrid gains shrink — see ablation rows with small `error_rate`.
- Regenerate course artifacts with `python scripts/build_case_study.py`; poster PDF via `pdflatex` or ship `poster/NLP_poster.pdf`.